## Step 1 : Create the embedding text

For the initial RAG prototype, the embedding text is constructed from four
important patent sections:

- Title
- Abstract
- Claims
- Summary

These fields are combined into a single `embedding_text` column.

The original patent fields are retained because they will later be useful
for displaying metadata and retrieved patent information.

In [1]:
import chromadb

print(chromadb.__version__)

client = chromadb.PersistentClient(
    path="../data/chroma_db"
)

print(client.list_collections())

1.5.9
[Collection(name=patent_rag), Collection(name=patent_rag_chunks)]


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

PyTorch: 2.14.0+cpu
CUDA: False


In [2]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Imports successful")

Imports successful


In [3]:
from datasets import load_from_disk

dataset = load_from_disk("../data/sample_dataset")

dataset

Parameter 'format_kwargs'={} of the transform datasets.arrow_dataset.Dataset.set_format couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


DatasetDict({
    train: Dataset({
        features: ['patent_number', 'decision', 'title', 'abstract', 'claims', 'background', 'summary', 'description', 'cpc_label', 'ipc_label', 'filing_date', 'patent_issue_date', 'date_published', 'examiner_id'],
        num_rows: 16153
    })
    validation: Dataset({
        features: ['patent_number', 'decision', 'title', 'abstract', 'claims', 'background', 'summary', 'description', 'cpc_label', 'ipc_label', 'filing_date', 'patent_issue_date', 'date_published', 'examiner_id'],
        num_rows: 9094
    })
})

## Step 1.2 — Convert the dataset to DataFrames

The HUPD dataset is stored as a Hugging Face `DatasetDict`.
We convert the train and validation splits into pandas DataFrames so that
the `embedding_text` column can be created and manipulated easily.

In [4]:
train_df = dataset["train"].to_pandas()
test_df = dataset["validation"].to_pandas()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (16153, 14)
Test shape: (9094, 14)


## Step 1.3 — Create the RAG corpus

For the initial RAG prototype, 1,000 patents are randomly sampled from
the training split.

A fixed random state is used so that the same corpus can be reproduced
when the notebook is run again.

In [5]:
rag_df = train_df.sample(n=1000, random_state=42).copy()

print("RAG corpus shape:", rag_df.shape)

RAG corpus shape: (1000, 14)


## Step 1.4 — Create the embedding text

For each patent in the RAG corpus, the title, abstract, and description
are combined into a single `embedding_text` field.

This field will be the text passed to the embedding model in the next stage.

In [6]:
text_columns = ["title", "abstract", "description"]

rag_df["embedding_text"] = (
    rag_df[text_columns]
    .fillna("")
    .agg("\n".join, axis=1)
)

## Step 1.5 — Inspect the embedding text

We inspect one patent's `embedding_text` to verify that the title,
abstract, and description were combined correctly before proceeding
to the embedding stage.

In [7]:
print(rag_df["embedding_text"].iloc[0])

SUPPORTING TOOL FOR SUPPORTING CYLINDRICAL PARTS, LIKE LANDING GEAR RODS AND CYLINDERS
Supporting tool (1) for supporting a substantially cylindrical part, like a landing gear rod or cylinder, said supporting tool comprising a metallic frame carrying at least one metallic ring (3) in electrical conductive relation with the frame, wherein the metallic ring carries a plurality of conical spikes (6) of non-conductive material projecting inward so as to engage the part with limited contact surface when said part is in place within the tool, said ring being openable so as to allow the part to be inserted into the ring and closable after the part has been inserted.
TECHNOLOGICAL BACKGROUND Supporting tools are known for supporting parts during specific treatments, like electrolytic coating. Such tools must be designed to support the part with minimal contacting surface to allow the near totality of the surface of the part to be accessible for treatment. However, such tools must also provide 

## Step 1.6 — Verify the embedding text column

Check that the new `embedding_text` column was added to the corpus
and that the original patent fields are still present.

In [8]:
print(rag_df.shape)
print(rag_df.columns.tolist())

(1000, 15)
['patent_number', 'decision', 'title', 'abstract', 'claims', 'background', 'summary', 'description', 'cpc_label', 'ipc_label', 'filing_date', 'patent_issue_date', 'date_published', 'examiner_id', 'embedding_text']


## Step 1.7 — Save the prepared RAG corpus

The 1,000-patent corpus, including the new `embedding_text` column,
is saved locally so that the prepared data can be reused without
reconstructing it.

In [9]:
rag_df.to_csv("../data/rag_corpus_1000.csv", index=False)

## Step 2 — Load the embedding model

The embedding model converts each patent's `embedding_text` into a
numerical vector representation.

These vectors will later be stored in ChromaDB and used for semantic
similarity search.

In [10]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


## Step 2.1 — Test the embedding model

Before generating embeddings for the full corpus, we test the model on
one patent.

The model converts the patent's `embedding_text` into a numerical vector.

In [11]:
test_embedding = model.encode(
    rag_df["embedding_text"].iloc[0]
)

print("Embedding shape:", test_embedding.shape)

Embedding shape: (384,)


## Step 2.2 — Generate embeddings for the RAG corpus

The embedding model is applied to the `embedding_text` of all 1,000 patents.

Each patent is converted into a 384-dimensional numerical vector.
These vectors will later be stored in ChromaDB for semantic retrieval.

In [12]:
embeddings = model.encode(
    rag_df["embedding_text"].tolist(),
    batch_size=32,
    show_progress_bar=True
)

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Embeddings shape: (1000, 384)


## Step 3 — Create the ChromaDB collection

ChromaDB will store the patent embeddings so that we can later perform
semantic similarity searches.

Each stored record will contain:
- the patent embedding
- the patent ID
- the patent text as the document
- relevant patent metadata

In [13]:
import sys

print(sys.executable)

/home/patzer_adi/Documents/patent_intelligence/.venv/bin/python


In [14]:
import chromadb

print("ChromaDB:", chromadb.__version__)

ChromaDB: 1.5.9


In [15]:
import chromadb

client = chromadb.PersistentClient(path="../data/chroma_db")

collection = client.get_or_create_collection(
    name="patent_rag"
)

print("ChromaDB collection created successfully.")

ChromaDB collection created successfully.


## Step 4 — Store patent embeddings in ChromaDB

Each patent will be stored with:

- its embedding vector
- its `embedding_text`
- basic metadata such as the application number and decision

This allows ChromaDB to retrieve the corresponding patent when a query is
submitted later.

In [16]:
collection.add(
    ids=rag_df["patent_number"].astype(str).tolist(),
    embeddings=embeddings.tolist(),
    documents=rag_df["embedding_text"].tolist(),
    metadatas=rag_df[
        ["patent_number", "decision", "title"]
    ].fillna("").to_dict(orient="records")
)

print("Patents stored:", collection.count())

Patents stored: 1000


## Step 5 — Semantic retrieval

A user query is converted into an embedding using the same embedding model.

ChromaDB then compares the query embedding with the stored patent embeddings
and returns the most semantically similar patents.

In [17]:
query = """
A device for holding cylindrical components during surface treatment
while minimizing the contact area with the component.
"""

query_embedding = model.encode(query)

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5,
    include=["embeddings", "documents", "metadatas"]
)

retrieved_embeddings = np.array(results["embeddings"][0])

similarities = cosine_similarity(
    query_embedding.reshape(1, -1),
    retrieved_embeddings
)[0]

print("Retrieved patents:\n")

for i, (patent_number, similarity) in enumerate(
    zip(results["ids"][0], similarities),
    start=1
):
    print(f"{i}. {patent_number}  |  Similarity: {similarity * 100:.4f}")

Retrieved patents:

1. 14904285  |  Similarity: 59.9075
2. 14995875  |  Similarity: 52.1716
3. 14904839  |  Similarity: 51.4235
4. 15001440  |  Similarity: 50.1430
5. 14904771  |  Similarity: 48.8613


In [18]:
print("Maximum sequence length:", model.max_seq_length)

Maximum sequence length: 256


In [19]:
print(model.tokenizer.model_max_length)

256


## Step 3.1 — Split patents into chunks

Since `all-MiniLM-L6-v2` has a maximum sequence length of 256 tokens,
the patent text is divided into smaller overlapping chunks.

Each chunk contains at most 200 tokens, with an overlap of 40 tokens
between consecutive chunks.

In [20]:
CHUNK_SIZE = 240
CHUNK_OVERLAP = 40

def chunk_text(text, tokenizer, chunk_size=240, overlap=40):
    tokens = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    start = 0

    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]

        chunk = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunks.append(chunk)

        if end >= len(tokens):
            break

        start = end - overlap

    return chunks

## Step 3.2 — Create chunks for the RAG corpus

The chunking function is applied to each patent's `embedding_text`.

Each patent may produce multiple chunks depending on the length of its text.

In [21]:
rag_df["chunks"] = rag_df["embedding_text"].apply(
    lambda text: chunk_text(
        text,
        model.tokenizer,
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP
    )
)

print("Chunks created successfully.")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1079 > 256). Running this sequence through the model will result in indexing errors


Chunks created successfully.


In [22]:
total_chunks = rag_df["chunks"].apply(len).sum()

print("Total chunks:", total_chunks)
print("Average chunks per patent:", rag_df["chunks"].apply(len).mean())

Total chunks: 67303
Average chunks per patent: 67.303


In [23]:
chunk_counts = rag_df["chunks"].apply(len)

print("Minimum chunks:", chunk_counts.min())
print("Maximum chunks:", chunk_counts.max())
print("Average chunks:", chunk_counts.mean())
print("Median chunks:", chunk_counts.median())

Minimum chunks: 6
Maximum chunks: 2933
Average chunks: 67.303
Median chunks: 46.0


In [24]:
max_idx = chunk_counts.idxmax()

print("Patent number:", rag_df.loc[max_idx, "patent_number"])
print("Character count:", len(rag_df.loc[max_idx, "embedding_text"]))
print("Chunk count:", chunk_counts.loc[max_idx])

Patent number: 14905310
Character count: 1166890
Chunk count: 2933


In [25]:
print(rag_df.loc[max_idx, "embedding_text"][:2000])


CYANOTRIAZOLE COMPOUNDS
This invention relates to a cyanotriazole compound represented by the formula (1): wherein each symbols are defined in the specification, or a salt thereof. The compound or a salt thereof stimulates the citric acid cycle activity and/or improves hyperglycemia with less side effects, and excellent safety, and therefore, it is useful for treating and/or preventing diseases or disorders on which citric acid cycle activation and/or improvement of hyperglycemia has a prophylactic and/or therapeutic effect, for example, diabetes, impaired glucose tolerance, insulin resistance, diabetic complications, obesity, dyslipidemia, hepatic steatosis, atherosclerosis and/or cardiovascular disease, as well as diseases or disorders that would benefit from stimulating energy expenditure.
TECHNICAL FIELD This invention relates to cyanotriazole compounds which stimulate the citric acid cycle activity and/or improve hyperglycemia. The citric acid cycle activators are useful for treat

In [26]:
text = rag_df.loc[max_idx, "embedding_text"]

tokens = model.tokenizer.encode(
    text,
    add_special_tokens=False
)

print("Token count:", len(tokens))

Token count: 586499


In [27]:
token_lengths = rag_df["embedding_text"].apply(
    lambda text: len(
        model.tokenizer.encode(
            text,
            add_special_tokens=False
        )
    )
)

print("Minimum tokens:", token_lengths.min())
print("Maximum tokens:", token_lengths.max())
print("Mean tokens:", token_lengths.mean())
print("Median tokens:", token_lengths.median())

Minimum tokens: 1042
Maximum tokens: 586499
Mean tokens: 13402.204
Median tokens: 9148.0


In [28]:
print("90th percentile:", token_lengths.quantile(0.90))
print("95th percentile:", token_lengths.quantile(0.95))
print("99th percentile:", token_lengths.quantile(0.99))

90th percentile: 24559.700000000004
95th percentile: 35401.7
99th percentile: 72470.18999999997


In [29]:
for limit in [35000, 50000, 75000, 100000]:
    count = (token_lengths > limit).sum()
    print(f"Above {limit:,} tokens: {count} patents")

Above 35,000 tokens: 52 patents
Above 50,000 tokens: 19 patents
Above 75,000 tokens: 10 patents
Above 100,000 tokens: 7 patents


## Step 3.3 — Estimate Embedding Memory

Before generating embeddings for all chunks, estimate how much memory the
resulting vectors will require.

In [30]:
num_chunks = total_chunks
embedding_dim = model.get_embedding_dimension()

bytes_per_vector = embedding_dim * 4  # float32 = 4 bytes
raw_memory_mb = num_chunks * bytes_per_vector / (1024 ** 2)

print("Number of chunks:", num_chunks)
print("Embedding dimension:", embedding_dim)
print(f"Raw embedding memory: {raw_memory_mb:.2f} MB")

Number of chunks: 67303
Embedding dimension: 384
Raw embedding memory: 98.59 MB


## Step 3.4 — Create the Chunk-Level Dataset

Convert the list of chunks for each patent into a separate row.

Each row will represent one chunk and retain the patent information needed
later for retrieval and identification.

In [31]:
chunk_rows = []

for _, row in rag_df.iterrows():
    for chunk_index, chunk in enumerate(row["chunks"]):
        chunk_rows.append({
            "patent_number": row["patent_number"],
            "chunk_index": chunk_index,
            "chunk_text": chunk,
            "decision": row["decision"],
            "title": row["title"]
        })

chunks_df = pd.DataFrame(chunk_rows)

print("Chunk-level dataset shape:", chunks_df.shape)
chunks_df.head()

Chunk-level dataset shape: (67303, 5)


,patent_number,chunk_index,chunk_text,decision,title
0,14904285,0,supporting tool for supporting cylindrical par...,PENDING,SUPPORTING TOOL FOR SUPPORTING CYLINDRICAL PAR...
1,14904285,1,the invention an object of the invention is to...,PENDING,SUPPORTING TOOL FOR SUPPORTING CYLINDRICAL PAR...
2,14904285,2,of said frame members. brief description of th...,PENDING,SUPPORTING TOOL FOR SUPPORTING CYLINDRICAL PAR...
3,14904285,3,only one is visible on the fig. 1. the central...,PENDING,SUPPORTING TOOL FOR SUPPORTING CYLINDRICAL PAR...
4,14904285,4,the hook 5a. the ring 3 is equipped with a plu...,PENDING,SUPPORTING TOOL FOR SUPPORTING CYLINDRICAL PAR...


## Step 3.5 — Generate Chunk Embeddings

Each chunk is converted into a 384-dimensional embedding using the same
`all-MiniLM-L6-v2` model used for the earlier prototype.

In [33]:
test_chunks = chunks_df["chunk_text"].iloc[:32].tolist()

test_embeddings = model.encode(
    test_chunks,
    batch_size=32,
    show_progress_bar=True
)

print("Test embeddings shape:", test_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Test embeddings shape: (32, 384)


In [34]:
import time

test_chunks = chunks_df["chunk_text"].iloc[:320].tolist()

start = time.time()

test_embeddings = model.encode(
    test_chunks,
    batch_size=32,
    show_progress_bar=True
)

elapsed = time.time() - start

print(f"Time for 320 chunks: {elapsed:.2f} seconds")
print(f"Approximate chunks/second: {320 / elapsed:.2f}")

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Time for 320 chunks: 8.64 seconds
Approximate chunks/second: 37.02


In [35]:
import numpy as np
from tqdm.auto import tqdm

chunk_texts = chunks_df["chunk_text"].tolist()

chunk_embeddings = []

for i in tqdm(
    range(0, len(chunk_texts), 32),
    desc="Embedding chunks",
    unit="chunks"
):
    batch = chunk_texts[i:i + 32]

    batch_embeddings = model.encode(
        batch,
        batch_size=32,
        show_progress_bar=False
    )

    chunk_embeddings.append(batch_embeddings)

chunk_embeddings = np.vstack(chunk_embeddings)

print("Embeddings shape:", chunk_embeddings.shape)

Embedding chunks:   0%|          | 0/2104 [00:00<?, ?chunks/s]

Embeddings shape: (67303, 384)


## Step 3.7 — Create the Chunk-Level ChromaDB Collection

A separate ChromaDB collection is created for the chunk-level embeddings.

Each record will contain:
- the chunk embedding
- the chunk text
- the patent number
- the chunk index
- the patent decision
- the patent title

In [36]:
import chromadb

client = chromadb.PersistentClient(path="../data/chroma_db")

chunk_collection = client.get_or_create_collection(
    name="patent_rag_chunks"
)

print("Chunk collection created successfully.")
print("Current count:", chunk_collection.count())

Chunk collection created successfully.
Current count: 0


## Step 3.8 — Store One Patent's Chunks in ChromaDB

Test storing all chunks belonging to a single patent.

Each chunk is stored as a separate ChromaDB record, while the
`patent_number` metadata keeps the chunks associated with the patent.

In [37]:
patent_number = chunks_df["patent_number"].iloc[0]

patent_chunks = chunks_df[
    chunks_df["patent_number"] == patent_number
]

print("Patent:", patent_number)
print("Number of chunks:", len(patent_chunks))

Patent: 14904285
Number of chunks: 6


In [38]:
chunk_embeddings_for_patent = chunk_embeddings[patent_chunks.index]

chunk_collection.add(
    ids=[
        f"{patent_number}_{idx}"
        for idx in patent_chunks["chunk_index"]
    ],
    embeddings=chunk_embeddings_for_patent.tolist(),
    documents=patent_chunks["chunk_text"].tolist(),
    metadatas=patent_chunks[
        ["patent_number", "chunk_index", "decision", "title"]
    ].fillna("").to_dict(orient="records")
)

print("Stored chunks:", len(patent_chunks))
print("ChromaDB count:", chunk_collection.count())

Stored chunks: 6
ChromaDB count: 6


In [39]:
chunk_collection.delete(
    ids=[
        f"{patent_number}_{idx}"
        for idx in patent_chunks["chunk_index"]
    ]
)

print("ChromaDB count:", chunk_collection.count())

ChromaDB count: 0


## Step 3.8 — Store Chunk Embeddings in ChromaDB

Each patent is processed separately. All chunks belonging to one patent are
stored together as individual ChromaDB records.

The total storage time is measured separately from the embedding generation.

In [40]:
import time

start_time = time.perf_counter()

patent_count = 0
total_patents = chunks_df["patent_number"].nunique()

for patent_number, patent_chunks in chunks_df.groupby("patent_number"):

    # Get the embeddings belonging to this patent
    patent_embeddings = chunk_embeddings[patent_chunks.index]

    # Create a unique ID for every chunk
    ids = [
        f"{patent_number}_{idx}"
        for idx in patent_chunks["chunk_index"]
    ]

    # Store this patent's chunks
    chunk_collection.add(
        ids=ids,
        embeddings=patent_embeddings.tolist(),
        documents=patent_chunks["chunk_text"].tolist(),
        metadatas=patent_chunks[
            ["patent_number", "chunk_index", "decision", "title"]
        ].fillna("").to_dict(orient="records")
    )

    patent_count += 1

    print(
        f"Patent {patent_count}/{total_patents} | "
        f"{patent_number} | "
        f"{len(patent_chunks)} chunks | "
        f"Stored: {chunk_collection.count()}"
    )

end_time = time.perf_counter()

elapsed = end_time - start_time

print("\nStorage complete.")
print(f"Total patents: {patent_count}")
print(f"Total chunks: {chunk_collection.count()}")
print(f"Storage time: {elapsed:.2f} seconds")
print(f"Storage time: {elapsed / 60:.2f} minutes")

Patent 1/1000 | 14348792 | 37 chunks | Stored: 37
Patent 2/1000 | 14654621 | 43 chunks | Stored: 80
Patent 3/1000 | 14779980 | 64 chunks | Stored: 144
Patent 4/1000 | 14784814 | 54 chunks | Stored: 198
Patent 5/1000 | 14786249 | 46 chunks | Stored: 244
Patent 6/1000 | 14888911 | 48 chunks | Stored: 292
Patent 7/1000 | 14889700 | 15 chunks | Stored: 307
Patent 8/1000 | 14889747 | 44 chunks | Stored: 351
Patent 9/1000 | 14895379 | 11 chunks | Stored: 362
Patent 10/1000 | 14897091 | 24 chunks | Stored: 386
Patent 11/1000 | 14902570 | 50 chunks | Stored: 436
Patent 12/1000 | 14902589 | 16 chunks | Stored: 452
Patent 13/1000 | 14902596 | 30 chunks | Stored: 482
Patent 14/1000 | 14902597 | 12 chunks | Stored: 494
Patent 15/1000 | 14902617 | 12 chunks | Stored: 506
Patent 16/1000 | 14902673 | 36 chunks | Stored: 542
Patent 17/1000 | 14902677 | 362 chunks | Stored: 904
Patent 18/1000 | 14902706 | 88 chunks | Stored: 992
Patent 19/1000 | 14902710 | 33 chunks | Stored: 1025
Patent 20/1000 | 1490

## Step 3.9 — Test Chunk-Level Semantic Retrieval

The same query used for the original patent-level retrieval is used again,
but this time the query is compared against the chunk-level embeddings
stored in ChromaDB.

This allows us to compare the retrieval results before and after chunking.

The top 5 retrieved chunks are examined along with their patent number,
chunk index, and cosine similarity.

In [45]:
query = """
A device for holding cylindrical components during surface treatment
while minimizing the contact area with the component.
"""

query_embedding = model.encode(query)

results = chunk_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=50,
    include=["embeddings", "documents", "metadatas"]
)

retrieved_embeddings = np.array(results["embeddings"][0])

similarities = cosine_similarity(
    query_embedding.reshape(1, -1),
    retrieved_embeddings
)[0]

print("Retrieved chunks:\n")

for i, (chunk_id, similarity, metadata) in enumerate(
    zip(
        results["ids"][0],
        similarities,
        results["metadatas"][0]
    ),
    start=1
):
    print(
        f"{i}. Patent: {metadata['patent_number']} "
        f"| Chunk: {metadata['chunk_index']} "
        f"| Similarity: {similarity:.4f}"
    )

Retrieved chunks:

1. Patent: 14904285 | Chunk: 0 | Similarity: 0.6053
2. Patent: 14989141 | Chunk: 6 | Similarity: 0.5762
3. Patent: 14904285 | Chunk: 1 | Similarity: 0.5748
4. Patent: 14988948 | Chunk: 19 | Similarity: 0.5650
5. Patent: 14904839 | Chunk: 27 | Similarity: 0.5638
6. Patent: 14988948 | Chunk: 18 | Similarity: 0.5562
7. Patent: 14992023 | Chunk: 40 | Similarity: 0.5528
8. Patent: 14904113 | Chunk: 6 | Similarity: 0.5402
9. Patent: 14989141 | Chunk: 50 | Similarity: 0.5380
10. Patent: 14904839 | Chunk: 5 | Similarity: 0.5343
11. Patent: 14904987 | Chunk: 14 | Similarity: 0.5291
12. Patent: 14989141 | Chunk: 58 | Similarity: 0.5281
13. Patent: 14989141 | Chunk: 4 | Similarity: 0.5280
14. Patent: 14905300 | Chunk: 23 | Similarity: 0.5279
15. Patent: 14996751 | Chunk: 12 | Similarity: 0.5218
16. Patent: 14905370 | Chunk: 7 | Similarity: 0.5184
17. Patent: 14904001 | Chunk: 197 | Similarity: 0.5172
18. Patent: 14993597 | Chunk: 25 | Similarity: 0.5146
19. Patent: 14990813 | C

In [46]:
patent_scores = {}

for metadata, similarity in zip(
    results["metadatas"][0],
    similarities
):
    patent_number = metadata["patent_number"]

    if patent_number not in patent_scores:
        patent_scores[patent_number] = []

    patent_scores[patent_number].append(similarity)

In [47]:
patent_scores = {
    patent: max(scores)
    for patent, scores in patent_scores.items()
}

In [48]:
ranked_patents = sorted(
    patent_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

print("Patent-level results:\n")

for rank, (patent_number, score) in enumerate(
    ranked_patents,
    start=1
):
    print(
        f"{rank}. {patent_number} | "
        f"Similarity: {score:.4f}"
    )

Patent-level results:

1. 14904285 | Similarity: 0.6053
2. 14989141 | Similarity: 0.5762
3. 14988948 | Similarity: 0.5650
4. 14904839 | Similarity: 0.5638
5. 14992023 | Similarity: 0.5528
6. 14904113 | Similarity: 0.5402
7. 14904987 | Similarity: 0.5291
8. 14905300 | Similarity: 0.5279
9. 14996751 | Similarity: 0.5218
10. 14905370 | Similarity: 0.5184
11. 14904001 | Similarity: 0.5172
12. 14993597 | Similarity: 0.5146
13. 14990813 | Similarity: 0.5104
14. 15001440 | Similarity: 0.5053
15. 14902717 | Similarity: 0.5042
16. 14991697 | Similarity: 0.5022
17. 14988231 | Similarity: 0.4997
18. 14988443 | Similarity: 0.4986
19. 14988720 | Similarity: 0.4976
20. 14993561 | Similarity: 0.4943
21. 15001644 | Similarity: 0.4929
22. 14903495 | Similarity: 0.4919
23. 14994259 | Similarity: 0.4891
24. 14993852 | Similarity: 0.4880
25. 14904771 | Similarity: 0.4866
26. 14993609 | Similarity: 0.4859
27. 14903825 | Similarity: 0.4834


In [49]:
print("Retrieved chunks:\n")

for i, (chunk_id, similarity, metadata) in enumerate(
    zip(
        results["ids"][0],
        similarities,
        results["metadatas"][0]
    ),
    start=1
):
    print(
        f"{i}. Patent: {metadata['patent_number']} "
        f"| Chunk: {metadata['chunk_index']} "
        f"| Similarity: {similarity:.4f}"
    )

Retrieved chunks:

1. Patent: 14904285 | Chunk: 0 | Similarity: 0.6053
2. Patent: 14989141 | Chunk: 6 | Similarity: 0.5762
3. Patent: 14904285 | Chunk: 1 | Similarity: 0.5748
4. Patent: 14988948 | Chunk: 19 | Similarity: 0.5650
5. Patent: 14904839 | Chunk: 27 | Similarity: 0.5638
6. Patent: 14988948 | Chunk: 18 | Similarity: 0.5562
7. Patent: 14992023 | Chunk: 40 | Similarity: 0.5528
8. Patent: 14904113 | Chunk: 6 | Similarity: 0.5402
9. Patent: 14989141 | Chunk: 50 | Similarity: 0.5380
10. Patent: 14904839 | Chunk: 5 | Similarity: 0.5343
11. Patent: 14904987 | Chunk: 14 | Similarity: 0.5291
12. Patent: 14989141 | Chunk: 58 | Similarity: 0.5281
13. Patent: 14989141 | Chunk: 4 | Similarity: 0.5280
14. Patent: 14905300 | Chunk: 23 | Similarity: 0.5279
15. Patent: 14996751 | Chunk: 12 | Similarity: 0.5218
16. Patent: 14905370 | Chunk: 7 | Similarity: 0.5184
17. Patent: 14904001 | Chunk: 197 | Similarity: 0.5172
18. Patent: 14993597 | Chunk: 25 | Similarity: 0.5146
19. Patent: 14990813 | C

## Step 3.9 — Inspect Top Chunk Retrievals

Before defining a patent-level similarity score, retrieve a larger number of
matching chunks.

The purpose is to observe how relevant chunks are distributed across patents.
A single patent may have multiple highly similar chunks, while another patent
may have only one.

The retrieved chunks will therefore be examined first, without applying a
patent-level aggregation method.

For this inspection, the top 50 chunks are retrieved.

In [51]:
n_results = 50

results = chunk_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=n_results,
    include=["embeddings", "metadatas", "documents"]
)

retrieved_embeddings = np.array(results["embeddings"][0])

similarities = cosine_similarity(
    query_embedding.reshape(1, -1),
    retrieved_embeddings
)[0]

print("Top 50 retrieved chunks:\n")

for i, (similarity, metadata) in enumerate(
    zip(
        similarities,
        results["metadatas"][0]
    ),
    start=1
):
    print(
        f"{i}. Patent: {metadata['patent_number']} "
        f"| Chunk: {metadata['chunk_index']} "
        f"| Similarity: {similarity:.4f}"
    )

Top 50 retrieved chunks:

1. Patent: 14904285 | Chunk: 0 | Similarity: 0.6053
2. Patent: 14989141 | Chunk: 6 | Similarity: 0.5762
3. Patent: 14904285 | Chunk: 1 | Similarity: 0.5748
4. Patent: 14988948 | Chunk: 19 | Similarity: 0.5650
5. Patent: 14904839 | Chunk: 27 | Similarity: 0.5638
6. Patent: 14988948 | Chunk: 18 | Similarity: 0.5562
7. Patent: 14992023 | Chunk: 40 | Similarity: 0.5528
8. Patent: 14904113 | Chunk: 6 | Similarity: 0.5402
9. Patent: 14989141 | Chunk: 50 | Similarity: 0.5380
10. Patent: 14904839 | Chunk: 5 | Similarity: 0.5343
11. Patent: 14904987 | Chunk: 14 | Similarity: 0.5291
12. Patent: 14989141 | Chunk: 58 | Similarity: 0.5281
13. Patent: 14989141 | Chunk: 4 | Similarity: 0.5280
14. Patent: 14905300 | Chunk: 23 | Similarity: 0.5279
15. Patent: 14996751 | Chunk: 12 | Similarity: 0.5218
16. Patent: 14905370 | Chunk: 7 | Similarity: 0.5184
17. Patent: 14904001 | Chunk: 197 | Similarity: 0.5172
18. Patent: 14993597 | Chunk: 25 | Similarity: 0.5146
19. Patent: 14990

## Step 3.10 — Group Retrieved Chunks by Patent

The retrieved chunks are grouped according to their patent number.

This allows multiple relevant chunks belonging to the same patent to be
considered together before calculating a patent-level relevance score.

In [52]:
from collections import defaultdict

patent_chunks = defaultdict(list)

for metadata, similarity in zip(
    results["metadatas"][0],
    similarities
):
    patent_number = metadata["patent_number"]

    patent_chunks[patent_number].append({
        "chunk_index": metadata["chunk_index"],
        "similarity": similarity
    })

print("Retrieved patents:", len(patent_chunks))

print("\nChunks grouped by patent:\n")

for patent_number, chunks in patent_chunks.items():
    print(
        f"Patent: {patent_number} | "
        f"Chunks retrieved: {len(chunks)}"
    )

Retrieved patents: 27

Chunks grouped by patent:

Patent: 14904285 | Chunks retrieved: 2
Patent: 14989141 | Chunks retrieved: 4
Patent: 14988948 | Chunks retrieved: 4
Patent: 14904839 | Chunks retrieved: 4
Patent: 14992023 | Chunks retrieved: 3
Patent: 14904113 | Chunks retrieved: 3
Patent: 14904987 | Chunks retrieved: 1
Patent: 14905300 | Chunks retrieved: 1
Patent: 14996751 | Chunks retrieved: 3
Patent: 14905370 | Chunks retrieved: 2
Patent: 14904001 | Chunks retrieved: 1
Patent: 14993597 | Chunks retrieved: 1
Patent: 14990813 | Chunks retrieved: 2
Patent: 15001440 | Chunks retrieved: 1
Patent: 14902717 | Chunks retrieved: 2
Patent: 14991697 | Chunks retrieved: 1
Patent: 14988231 | Chunks retrieved: 3
Patent: 14988443 | Chunks retrieved: 1
Patent: 14988720 | Chunks retrieved: 1
Patent: 14993561 | Chunks retrieved: 1
Patent: 15001644 | Chunks retrieved: 2
Patent: 14903495 | Chunks retrieved: 1
Patent: 14994259 | Chunks retrieved: 2
Patent: 14993852 | Chunks retrieved: 1
Patent: 149047

## Step 3.11 — Extract Strongest Chunk Similarities per Patent

For each candidate patent, the retrieved chunk similarities are sorted
from highest to lowest.

We retain:

- the maximum similarity
- the top 3 chunk similarities

These values will later be combined into a single patent-level
relevance score.

In [53]:
patent_evidence = {}

for patent_number, chunks in patent_chunks.items():

    similarities_for_patent = sorted(
        [chunk["similarity"] for chunk in chunks],
        reverse=True
    )

    patent_evidence[patent_number] = {
        "max_similarity": similarities_for_patent[0],
        "top_3_similarities": similarities_for_patent[:3]
    }

print("Patents with evidence:", len(patent_evidence))

Patents with evidence: 27


## Step 3.12 — Inspect Patent-Level Evidence

The strongest chunk similarity and the top retrieved chunk similarities
are displayed for each candidate patent.

This allows us to examine how consistently each patent matches the query
before defining the final patent-level scoring formula.

In [54]:
for patent_number, evidence in patent_evidence.items():
    print(
        f"Patent: {patent_number} | "
        f"Max: {evidence['max_similarity']:.4f} | "
        f"Top-3: {[round(s, 4) for s in evidence['top_3_similarities']]}"
    )

Patent: 14904285 | Max: 0.6053 | Top-3: [np.float64(0.6053), np.float64(0.5748)]
Patent: 14989141 | Max: 0.5762 | Top-3: [np.float64(0.5762), np.float64(0.538), np.float64(0.5281)]
Patent: 14988948 | Max: 0.5650 | Top-3: [np.float64(0.565), np.float64(0.5562), np.float64(0.4948)]
Patent: 14904839 | Max: 0.5638 | Top-3: [np.float64(0.5638), np.float64(0.5343), np.float64(0.5072)]
Patent: 14992023 | Max: 0.5528 | Top-3: [np.float64(0.5528), np.float64(0.5032), np.float64(0.4932)]
Patent: 14904113 | Max: 0.5402 | Top-3: [np.float64(0.5402), np.float64(0.5041), np.float64(0.5009)]
Patent: 14904987 | Max: 0.5291 | Top-3: [np.float64(0.5291)]
Patent: 14905300 | Max: 0.5279 | Top-3: [np.float64(0.5279)]
Patent: 14996751 | Max: 0.5218 | Top-3: [np.float64(0.5218), np.float64(0.5006), np.float64(0.4811)]
Patent: 14905370 | Max: 0.5184 | Top-3: [np.float64(0.5184), np.float64(0.4851)]
Patent: 14904001 | Max: 0.5172 | Top-3: [np.float64(0.5172)]
Patent: 14993597 | Max: 0.5146 | Top-3: [np.float64

## Step 3.13 — Calculate the Combined Patent-Level Score

Each patent receives a single relevance score based on two signals:

1. Maximum chunk similarity — captures the strongest evidence.
2. Mean similarity of the top 3 retrieved chunks — captures consistency
   across multiple relevant chunks.

The two signals are weighted equally:

Patent Score = 0.5 × Max Similarity + 0.5 × Top-3 Mean Similarity

For patents with fewer than three retrieved chunks, all available
chunks are used.

In [56]:
patent_scores = {}

for patent_number, evidence in patent_evidence.items():

    max_similarity = evidence["max_similarity"]

    top_3_mean = sum(
        evidence["top_3_similarities"]
    ) / len(evidence["top_3_similarities"])

    patent_score = (
        0.5 * max_similarity +
        0.5 * top_3_mean
    )

    patent_scores[patent_number] = {
        "max_similarity": max_similarity,
        "top_3_mean": top_3_mean,
        "patent_score": patent_score
    }

print("Patent-level scores calculated:", len(patent_scores))

Patent-level scores calculated: 27


## Step 3.14 — Rank Patents by Combined Score

The candidate patents are ranked according to their combined
patent-level relevance score.

This produces a patent-level ranking rather than a chunk-level ranking.

In [57]:
ranked_patents = sorted(
    patent_scores.items(),
    key=lambda x: x[1]["patent_score"],
    reverse=True
)

for rank, (patent_number, scores) in enumerate(ranked_patents, start=1):
    print(
        f"{rank}. Patent: {patent_number} | "
        f"Score: {scores['patent_score']:.4f} | "
        f"Max: {scores['max_similarity']:.4f} | "
        f"Top-3 Mean: {scores['top_3_mean']:.4f}"
    )

1. Patent: 14904285 | Score: 0.5977 | Max: 0.6053 | Top-3 Mean: 0.5901
2. Patent: 14989141 | Score: 0.5618 | Max: 0.5762 | Top-3 Mean: 0.5474
3. Patent: 14988948 | Score: 0.5518 | Max: 0.5650 | Top-3 Mean: 0.5387
4. Patent: 14904839 | Score: 0.5495 | Max: 0.5638 | Top-3 Mean: 0.5351
5. Patent: 14992023 | Score: 0.5346 | Max: 0.5528 | Top-3 Mean: 0.5164
6. Patent: 14904987 | Score: 0.5291 | Max: 0.5291 | Top-3 Mean: 0.5291
7. Patent: 14905300 | Score: 0.5279 | Max: 0.5279 | Top-3 Mean: 0.5279
8. Patent: 14904113 | Score: 0.5276 | Max: 0.5402 | Top-3 Mean: 0.5151
9. Patent: 14904001 | Score: 0.5172 | Max: 0.5172 | Top-3 Mean: 0.5172
10. Patent: 14993597 | Score: 0.5146 | Max: 0.5146 | Top-3 Mean: 0.5146
11. Patent: 14996751 | Score: 0.5115 | Max: 0.5218 | Top-3 Mean: 0.5012
12. Patent: 14905370 | Score: 0.5100 | Max: 0.5184 | Top-3 Mean: 0.5017
13. Patent: 14990813 | Score: 0.5075 | Max: 0.5104 | Top-3 Mean: 0.5047
14. Patent: 15001440 | Score: 0.5053 | Max: 0.5053 | Top-3 Mean: 0.5053
1